# Accessing Data with APIs

## From a request to an exploratory analysis

### Learning objectives

By the end of this lesson, you should be able to:

- explain what an application programming interface (API) does;
- read basic API documentation and identify an endpoint, parameters, and response format;
- make a `GET` request with Python's `requests` library;
- check a response before using it;
- turn JSON records into a pandas DataFrame;
- clean dates, missing values, and numeric columns returned by an API;
- summarize and visualize API data with pandas and Seaborn;
- keep API keys and passwords out of notebooks;
- distinguish open APIs from subscription databases such as WRDS.

Throughout the notebook, stop at the **Try it** and **Exploration problem** prompts before moving on.

In [ ]:
import os
from getpass import getpass

import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 1. What is an API?

An **application programming interface** provides a structured way for one program to request data or services from another. For a web API, a request usually contains:

- an **endpoint**: the URL identifying the resource;
- **parameters**: choices such as a series, country, date range, or output format;
- sometimes an **API key** or another form of authentication.

The server sends a **response** containing:

- a status code, such as `200` for success or `404` for not found;
- headers with information about the response;
- a body, often encoded as JSON.

APIs let us request only what we need, but they also impose responsibilities. Read the documentation, request a reasonable amount of data, respect rate limits and terms of use, and document the source and retrieval date.

### Common response codes

| Code | Meaning | Likely next step |
|---:|---|---|
| 200 | Success | Inspect and parse the response |
| 400 | Bad request | Check the endpoint and parameters |
| 401/403 | Authentication or permission problem | Check credentials and access rights |
| 404 | Resource not found | Check the URL or identifier |
| 429 | Too many requests | Pause and respect the rate limit |
| 500+ | Server problem | Retry later; do not assume your code is wrong |

`response.raise_for_status()` converts an unsuccessful HTTP response into a visible Python error instead of allowing later code to fail in a confusing way.

### Protect credentials

Never place a real API key, password, or token directly in a notebook. Notebooks are frequently shared, uploaded, or committed to Git. Use an environment variable or a hidden prompt instead.

If a key has ever been included in a shared notebook, revoke it and create a new one.

## 2. FRED: an authenticated REST API

The Federal Reserve Bank of St. Louis provides the [FRED API](https://fred.stlouisfed.org/docs/api/fred/). You need a free FRED API key for this section.

The next cell first looks for a `FRED_API_KEY` environment variable. If none exists, it prompts for the key without displaying it. The key remains in memory only for this notebook session.

In [ ]:
FRED_API_KEY = os.getenv("FRED_API_KEY")

if not FRED_API_KEY:
    FRED_API_KEY = getpass("Enter your FRED API key: ")

print("FRED key loaded:", bool(FRED_API_KEY))

### Request a list of data releases

Pass parameters through the `params` argument instead of manually assembling a long URL. This is easier to read and lets `requests` encode the values correctly.

In [ ]:
fred_releases_url = "https://api.stlouisfed.org/fred/releases"
release_params = {
    "api_key": FRED_API_KEY,
    "file_type": "json",
}

release_response = requests.get(
    fred_releases_url,
    params=release_params,
    timeout=30,
)
release_response.raise_for_status()

print("Status:", release_response.status_code)
print("Content type:", release_response.headers.get("content-type"))

In [ ]:
release_json = release_response.json()
release_json.keys()

In [ ]:
releases = pd.DataFrame(release_json["releases"])
releases.head()

### Try it 1 — Inspect the response (4–6 minutes)

1. How many releases were returned?
2. Which columns does the DataFrame contain?
3. Sort the releases by name.
4. Find releases whose names contain `employment`, ignoring capitalization.
5. What is the unit of observation in `releases`?

In [ ]:
# Try it 1

### Search for a FRED series

The [series search endpoint](https://fred.stlouisfed.org/docs/api/fred/series_search.html) accepts a text query. Notice that the search phrase is a parameter, not part of the endpoint itself.

In [ ]:
fred_search_url = "https://api.stlouisfed.org/fred/series/search"
search_params = {
    "search_text": "consumer price index",
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "limit": 20,
}

search_response = requests.get(fred_search_url, params=search_params, timeout=30)
search_response.raise_for_status()
search_json = search_response.json()

series_results = pd.DataFrame(search_json["seriess"])
series_results[["id", "title", "frequency", "units", "seasonal_adjustment"]].head(10)

### Try it 2 — Compare search results (5–8 minutes)

Search for a different economic concept such as wages, mortgage rates, industrial production, or consumer sentiment.

- Keep the result to no more than 20 records.
- Display the series ID, title, frequency, units, and seasonal adjustment.
- Select one series and explain why it is appropriate for your question.

Be careful: similarly titled series may use different units, frequencies, populations, or seasonal adjustments.

In [ ]:
# Try it 2
my_search_term = ""

### Retrieve and clean a time series

`UNRATE` is the monthly U.S. civilian unemployment rate. API values arrive as strings, and FRED sometimes represents missing observations with a period (`.`). We therefore convert values with `errors="coerce"`, which turns nonnumeric values into `NaN`.

In [ ]:
fred_observations_url = "https://api.stlouisfed.org/fred/series/observations"
observation_params = {
    "series_id": "UNRATE",
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": "2000-01-01",
}

unrate_response = requests.get(
    fred_observations_url,
    params=observation_params,
    timeout=30,
)
unrate_response.raise_for_status()
unrate_json = unrate_response.json()

unrate = pd.DataFrame(unrate_json["observations"])
unrate.head()

In [ ]:
unrate = unrate[["date", "value"]].copy()
unrate["date"] = pd.to_datetime(unrate["date"])
unrate["value"] = pd.to_numeric(unrate["value"], errors="coerce")

unrate.info()
display(unrate.isna().sum())
unrate.head()

In [ ]:
plt.figure(figsize=(11, 5))
sns.lineplot(data=unrate, x="date", y="value")
plt.title("U.S. Civilian Unemployment Rate")
plt.xlabel("Date")
plt.ylabel("Unemployment rate (%)")
plt.tight_layout()
plt.show()

### Exploration problem 1 — A FRED series (10–15 minutes)

Choose another FRED series and create a small exploratory analysis.

1. Record the series ID, title, units, frequency, and seasonal-adjustment status.
2. Request a manageable date range.
3. Convert the date and value columns to appropriate data types.
4. Check missing values and calculate a useful summary.
5. Create a clearly labeled visualization.
6. Write two observations and one question requiring additional context.

Do not infer causation from a single time-series graph.

In [ ]:
# Exploration problem 1: request and clean your series.

In [ ]:
# Exploration problem 1: summarize and visualize your series.

**Series documentation and interpretation:**

>

## 3. World Bank Indicators: a public API without a key

The [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589-developer-information) provides development indicators and metadata. It is useful as a fallback when students do not have credentials for FRED or WRDS.

This request compares constant-price GDP per person for Canada, Mexico, and the United States. World Bank JSON responses contain metadata in the first list element and the observations in the second.

In [ ]:
wb_url = (
    "https://api.worldbank.org/v2/country/CAN;MEX;USA/"
    "indicator/NY.GDP.PCAP.KD"
)
wb_params = {
    "format": "json",
    "date": "2000:2024",
    "per_page": 1000,
}

wb_response = requests.get(wb_url, params=wb_params, timeout=30)
wb_response.raise_for_status()
wb_json = wb_response.json()

print("Metadata:", wb_json[0])
print("Number of observations:", len(wb_json[1]))

In [ ]:
wb_gdp = pd.json_normalize(wb_json[1])
wb_gdp = wb_gdp[["country.value", "countryiso3code", "date", "value"]].rename(
    columns={
        "country.value": "country",
        "countryiso3code": "country_code",
        "date": "year",
        "value": "gdp_per_capita",
    }
)
wb_gdp["year"] = pd.to_numeric(wb_gdp["year"])
wb_gdp["gdp_per_capita"] = pd.to_numeric(wb_gdp["gdp_per_capita"], errors="coerce")
wb_gdp = wb_gdp.sort_values(["country", "year"])

wb_gdp.head()

In [ ]:
display(wb_gdp.isna().sum())
display(wb_gdp.groupby("country", observed=True)["gdp_per_capita"].describe().round(1))

plt.figure(figsize=(11, 5))
sns.lineplot(data=wb_gdp, x="year", y="gdp_per_capita", hue="country", marker="o")
plt.title("GDP per Person in Canada, Mexico, and the United States")
plt.xlabel("Year")
plt.ylabel("GDP per person (constant 2015 US$)")
plt.tight_layout()
plt.show()

### Exploration problem 2 — Compare countries or indicators (12–18 minutes)

Use the World Bank API to investigate a question of your choice.

Possible indicators:

- `SP.POP.TOTL` — total population
- `SL.UEM.TOTL.ZS` — unemployment rate
- `NE.TRD.GNFS.ZS` — trade as a percentage of GDP
- `SE.XPD.TOTL.GD.ZS` — government education expenditure as a percentage of GDP
- `EN.ATM.CO2E.PC` — carbon dioxide emissions per person

Requirements:

1. Compare at least two countries or two indicators.
2. Inspect the metadata and missing values.
3. Use filtering or `groupby` to produce a useful summary.
4. Create a Seaborn visualization.
5. Explain what the graph shows, what it does not show, and one reason the comparison may be misleading.

In [ ]:
# Exploration problem 2: construct the request and DataFrame.

In [ ]:
# Exploration problem 2: summarize and visualize.

**Interpretation:**

>

## 4. SEC EDGAR: responsible access to company data

The SEC provides public [EDGAR data APIs](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) for company submissions and XBRL company facts. These endpoints do not require an API key, but automated requests must follow the SEC's fair-access guidance.

Send a descriptive `User-Agent` containing your name and email. Replace the placeholder below before making a request. Do not send rapid repeated requests.

The example uses Apple's CIK, padded to ten digits, and retrieves company facts. A company may report similar concepts under different tags or units, so financial data still requires careful interpretation.

In [ ]:
SEC_USER_AGENT = "Your Name your.email@example.com"

if "example.com" in SEC_USER_AGENT:
    print("Replace SEC_USER_AGENT with your real name and email before continuing.")

In [ ]:
apple_cik = "0000320193"
companyfacts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{apple_cik}.json"
sec_headers = {"User-Agent": SEC_USER_AGENT}

# Run after replacing SEC_USER_AGENT above.
# sec_response = requests.get(companyfacts_url, headers=sec_headers, timeout=30)
# sec_response.raise_for_status()
# companyfacts = sec_response.json()
# companyfacts.keys()

### Exploration problem 3 — Inspect company facts (10–15 minutes)

After your request succeeds:

1. Display the company name and available taxonomy keys.
2. Inspect `companyfacts["facts"]["us-gaap"].keys()`.
3. Find a concept such as `Revenues`, `RevenueFromContractWithCustomerExcludingAssessedTax`, `Assets`, or `NetIncomeLoss`.
4. Convert one concept's `units["USD"]` records to a DataFrame.
5. Filter to annual 10-K values and inspect duplicates before plotting.
6. Create a time-series plot and explain why multiple filings may report the same fiscal period.

**Important:** accounting concepts and company reporting practices vary. Do not assume one tag works identically for every company.

In [ ]:
# Exploration problem 3
# Example starting point after companyfacts has been created:
# gaap_facts = companyfacts["facts"]["us-gaap"]
# list(gaap_facts.keys())[:20]

In [ ]:
# Convert one selected concept to a DataFrame, filter it, and plot it.

## 5. WRDS: optional institutional database access

[Wharton Research Data Services (WRDS)](https://wrds-www.wharton.upenn.edu/) provides a common research platform for many financial, accounting, banking, and market datasets. Unlike the previous public APIs, WRDS requires an account, institutional authorization, and access to the particular database being queried.

The Python package connects to WRDS's database service. Do **not** type a password into a visible code cell. The connection process prompts securely when credentials are needed.

If your account, campus network, VPN, or subscription does not permit access, complete the **World Bank alternative** below instead.

In [ ]:
# Uncomment once per environment if the package is not installed.
# %pip install wrds

In [ ]:
# Optional WRDS connection
# import wrds
# wrds_username = input("WRDS username: ")
# db = wrds.Connection(wrds_username=wrds_username)

### Learn the database before querying it

WRDS hosts many libraries, and access varies by institution. Begin by listing the libraries you can use. Then list tables and describe the table you plan to query.

Do not begin with `select *` on a large research table. Select only needed columns, constrain dates, and use a small test query.

In [ ]:
# Run only after connecting.
# libraries = db.list_libraries()
# libraries[:20]

In [ ]:
# Compustat examples; these require a Compustat subscription.
# comp_tables = db.list_tables(library="comp")
# comp_tables[:20]

# funda_description = db.describe_table(library="comp", table="funda")
# funda_description.head(20)

### A small Compustat query

The following query requests a limited set of annual accounting variables for three firms and five fiscal years. The filters select standardized, consolidated, domestic-source industrial-format records. Database coverage and variable definitions should be checked in WRDS documentation before interpreting results.

In [ ]:
# Run only with authorized Compustat access.
# sql = """
# select gvkey, datadate, fyear, tic, conm, at, sale, ni
# from comp.funda
# where fyear between 2019 and 2023
#   and tic in ('AAPL', 'IBM', 'MSFT')
#   and indfmt = 'INDL'
#   and datafmt = 'STD'
#   and popsrc = 'D'
#   and consol = 'C'
# order by tic, fyear
# """
#
# fundamentals = db.raw_sql(sql, date_cols=["datadate"])
# fundamentals.head()

### Exploration problem 4A — WRDS company fundamentals (15–25 minutes)

If you have WRDS and Compustat access:

1. Confirm the table's variable names with `describe_table`.
2. Run the small query above; do not expand it to the full database.
3. Check shape, data types, duplicates, and missing values.
4. Create `profit_margin = ni / sale`, replacing infinite values with missing values if necessary.
5. Use `groupby` to summarize sales or profit margin by company.
6. Create a line plot comparing companies across fiscal years.
7. Explain two findings and two limitations of comparing these companies.

Close the connection when finished with `db.close()`.

In [ ]:
# Exploration problem 4A: inspect and prepare WRDS data.

In [ ]:
# Exploration problem 4A: summarize and visualize.

In [ ]:
# Close the connection when all WRDS work is finished.
# db.close()

### Exploration problem 4B — Public-data alternative

If WRDS is unavailable, use the World Bank API to compare one indicator across at least four countries over ten or more years.

Complete the same analytical steps:

1. inspect missingness and types;
2. create a relevant derived variable or change measure;
3. use `groupby` to summarize countries;
4. create a comparison plot;
5. write two findings and two limitations.

State that you completed the public-data alternative.

In [ ]:
# Exploration problem 4B

## 6. Final API investigation

Choose FRED, the World Bank, SEC EDGAR, WRDS, or another documented API approved by the instructor. Develop a small, reproducible investigation.

### Required components

1. **Question:** Write one focused question before requesting data.
2. **Documentation:** Link to the endpoint documentation and identify any access requirements.
3. **Request:** Use an endpoint and parameters to request a reasonable amount of data.
4. **Validation:** Check the status code and response structure.
5. **DataFrame:** Convert relevant records into a tidy pandas DataFrame.
6. **Cleaning:** Address dates, numeric types, and missing values.
7. **Summary:** Use selection, filtering, and/or `groupby`.
8. **Visualization:** Create at least one purposeful, labeled Seaborn graph.
9. **Interpretation:** State two findings, one limitation, and one next step.
10. **Reproducibility:** Remove credentials and confirm that another authorized user could rerun the notebook.

### Planning notes

- **API/database:**
- **Documentation URL:**
- **Access requirements:**
- **Unit of observation:**
- **Question:**
- **Fields or variables needed:**
- **Expected limitation:**

In [ ]:
# Final investigation: request and validation

In [ ]:
# Final investigation: DataFrame construction and cleaning

In [ ]:
# Final investigation: summary and visualization

### Interpretation

**Finding 1:**  

**Finding 2:**  

**Limitation:**  

**Next step:**  

**What this analysis does not establish:**

## Exit check

Before finishing, answer briefly:

1. What is the difference between an endpoint and a parameter?
2. Why should you call `raise_for_status()`?
3. Why might numeric data arrive as strings?
4. Where should an API key be stored?
5. Why should a WRDS query begin with a small number of columns, years, and firms?
6. What information must accompany a visualization made from API data?

## Documentation and further reading

- [FRED API documentation](https://fred.stlouisfed.org/docs/api/fred/)
- [World Bank Indicators API documentation](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589-developer-information)
- [SEC EDGAR data APIs](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)
- [SEC developer resources and fair-access guidance](https://www.sec.gov/about/developer-resources)
- [WRDS: Using Python](https://wrds-www.wharton.upenn.edu/pages/grid-items/using-python-wrds-platform/)
- [WRDS: Using Jupyter](https://wrds-www.wharton.upenn.edu/pages/classroom/using-jupyter-at-wrds-notebook/)
- [`requests` documentation](https://requests.readthedocs.io/)
- [`pandas.json_normalize` documentation](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)